# Pilot-0 existing-data audit
Executed locally from saved summary data; no remote calls. Full report follows.


In [1]:
import json
from pathlib import Path
p = Path('summary.json')
if not p.exists(): p = Path('artifacts/replay-v1/pilot-audit/summary.json')
data = json.loads(p.read_text())
for pair in data['pairs']:
    print('Pair', pair['pair'], 'gain = absolute − baseline:', pair['decomposition'])


Pair 1 gain = absolute − baseline: [{'window': [0, 40], 'absolute_difference': -0.009265136718749997, 'baseline_difference': 0.0499267578125, 'gain_difference': -0.05919189453125, 'identity_residual': 0.0}, {'window': [0, 480], 'absolute_difference': -0.010568237304687533, 'baseline_difference': 0.0499267578125, 'gain_difference': -0.06049499511718753, 'identity_residual': 0.0}]
Pair 2 gain = absolute − baseline: [{'window': [0, 40], 'absolute_difference': 0.0025497436523437417, 'baseline_difference': 0.053466796875, 'gain_difference': -0.05091705322265626, 'identity_residual': 0.0}, {'window': [0, 480], 'absolute_difference': 0.11391080220540364, 'baseline_difference': 0.053466796875, 'gain_difference': 0.060444005330403644, 'identity_residual': 0.0}]


# Pilot-0 existing-data audit

Descriptive, post-hoc analysis of the two completed pairs. No new sampling, training, verifier relaxation, or gate changes. Original Pilot reports and uncertainty settings remain intact.

## Reproduce before extending

| Pair | Stored F1/F2 cells | Original intervals | Maximum interval discrepancy |
| --- | --- | --- | --- |
| 1 | exact | 14 | 0.0 |
| 2 | exact | 14 | 0.0 |

All new contrasts below are **B-G minus B-S**. The reproduced historical intervals in `summary.json` retain their original **B-S minus B-G** direction. Raw Pass@1 is unconditional exact successes divided by generated completions; all failures stay in the denominator. Posterior scores are item-averaged Jeffreys means and are kept separately for provenance.

New pointwise 95% intervals use 50,000 paired item-trajectory resamples. NumPy PCG64 receives the unsigned big-endian integer from the first eight SHA256 digest bytes of `duraseed-replay-v1|pilot-audit|<contrast>`. Contrast names and resulting seeds are retained in JSON. Each item retains both arms, its whole trajectory, and all realized within-item draws. The conditional intervals do not include training-run, checkpoint-selection, or fresh-completion uncertainty; no pooling across the two source blocks is performed. No optional family-block sensitivity was added.

AUC integrates the linear checkpoint grid and divides by the window width. The first-attainment thresholds 0.06, 0.07, 0.10, 0.20 are post-hoc sensitivities for Pilot 0, fixed before this calculation. Crossing and half-life summaries below are descriptive point estimates, not fitted decay constants or observed intermediate checkpoints; no bootstrap non-crossings were discarded to form an interval.

![Absolute and own-baseline-relative learning](learning.svg)

![Retention and chronological downstream-performance paths](retention.svg)

![Disjoint early verifier outcomes](failures.svg)

## Pair 1

### Absolute performance, gain, and endpoint

| Arm | Baseline | Absolute AUC 0–40 | Gain AUC 0–40 | Absolute AUC 0–480 | Gain AUC 0–480 | Endpoint 480 |
| --- | --- | --- | --- | --- | --- | --- |
| B-S | 0.00000000 | 0.07692719 | 0.07692719 | 0.26229541 | 0.26229541 | 0.37817383 |
| B-G | 0.04992676 | 0.06766205 | 0.01773529 | 0.25172717 | 0.20180041 | 0.40368652 |

Gain contrast = absolute-AUC contrast − baseline contrast:

| Window | Absolute difference | Baseline difference | Gain difference | Numerical residual |
| --- | --- | --- | --- | --- |
| [0, 40] | -0.00926514 | 0.04992676 | -0.05919189 | 0 |
| [0, 480] | -0.01056824 | 0.04992676 | -0.06049500 | 0 |

### Unconditional retention

| Arm | Role | Raw baseline | Absolute AUC 0–20 | Own-baseline-relative AUC | Half-life | Bracket/status |
| --- | --- | --- | --- | --- | --- | --- |
| B-S | targeted | 0.29817708 | 0.05458984 | 0.18307860 | 2.66428571 | [2, 5] |
| B-S | sentinel | 0.04427083 | 0.01357422 | 0.30661765 | 6.31578947 | [5, 10] |
| B-G | targeted | 0.30859375 | 0.07425130 | 0.24061181 | 4.10465116 | [2, 5] |
| B-G | sentinel | 0.29557292 | 0.07587891 | 0.25671806 | 4.11071429 | [2, 5] |

These scores are total observed pre-B performance, not survival restricted to newly acquired items.

### Paired item uncertainty

| Contrast | Estimate | 95% lower | 95% upper | Items |
| --- | --- | --- | --- | --- |
| pair1-seed11-targeted-absolute-AUC-0-20-B-G-minus-B-S | 0.01966146 | 0.00937500 | 0.03017578 | 192 |
| pair1-seed11-sentinel-absolute-AUC-0-20-B-G-minus-B-S | 0.06230469 | 0.05485026 | 0.06985677 | 192 |
| pair1-seed11-maps-absolute-AUC-0-40-B-G-minus-B-S | -0.00926514 | -0.01230164 | -0.00627899 | 512 |
| pair1-seed11-maps-gain-AUC-0-40-B-G-minus-B-S | -0.05919189 | -0.06694798 | -0.05182190 | 512 |
| pair1-seed11-maps-absolute-AUC-0-480-B-G-minus-B-S | -0.01056824 | -0.01943810 | -0.00168246 | 512 |
| pair1-seed11-maps-gain-AUC-0-480-B-G-minus-B-S | -0.06049500 | -0.07163046 | -0.04974426 | 512 |
| pair1-maps-baseline-B-G-minus-B-S | 0.04992676 | 0.04357910 | 0.05651855 | 512 |
| pair1-maps-endpoint480-B-G-minus-B-S | 0.02551270 | 0.00793457 | 0.04357910 | 512 |

### First attainment at comparable absolute MAPS performance

| Threshold | Arm | Status | Interpolated update | Observed bracket | Targeted TCES | Eligible B-G−B-S |
| --- | --- | --- | --- | --- | --- | --- |
| 0.06 | B-S | crossed | 4.70841629 | [2, 5] | 0.08698388 | — |
| 0.06 | B-G | crossed | 0.66016000 | [0, 1] | 0.29312125 | 0.20613737 |
| 0.07 | B-S | crossed | 9.31714286 | [5, 10] | 0.01516667 | — |
| 0.07 | B-G | crossed | 9.34814815 | [5, 10] | 0.02037423 | 0.00520756 |
| 0.1 | B-S | crossed | 32.85263158 | [20, 40] | 0.00130208 | — |
| 0.1 | B-G | crossed | 51.15092025 | [40, 80] | 0.00672354 | 0.00542146 |
| 0.2 | B-S | crossed | 120.38731219 | [80, 160] | 0.00195943 | — |
| 0.2 | B-G | crossed | 182.18003273 | [160, 320] | 0.00000000 | -0.00195943 |

`baseline_exceeded` is not equivalent learning progress. A two-arm post-training difference is printed only when both arms have a first crossing from below. Interpolation uses the same adjacent interval for update and retention and never extrapolates or erases a reversal.

### Full raw and posterior trajectories

| Panel/role | Arm | Update | Raw Pass@1 | Jeffreys posterior | MAPS raw gain |
| --- | --- | --- | --- | --- | --- |
| targeted | B-S | 0 | 0.29817708 | 0.33854167 | — |
| targeted | B-S | 1 | 0.24609375 | 0.29687500 | — |
| targeted | B-S | 2 | 0.16927083 | 0.23541667 | — |
| targeted | B-S | 5 | 0.07812500 | 0.16250000 | — |
| targeted | B-S | 10 | 0.00520833 | 0.10416667 | — |
| targeted | B-S | 20 | 0.00130208 | 0.10104167 | — |
| targeted | B-S | 40 | 0.00130208 | 0.10104167 | — |
| targeted | B-S | 80 | 0.00130208 | 0.10104167 | — |
| targeted | B-S | 160 | 0.00260417 | 0.10208333 | — |
| targeted | B-S | 320 | 0.00000000 | 0.10000000 | — |
| targeted | B-S | 480 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 0 | 0.30859375 | 0.34687500 | — |
| targeted | B-G | 1 | 0.28515625 | 0.32812500 | — |
| targeted | B-G | 2 | 0.27213542 | 0.31770833 | — |
| targeted | B-G | 5 | 0.10416667 | 0.18333333 | — |
| targeted | B-G | 10 | 0.00781250 | 0.10625000 | — |
| targeted | B-G | 20 | 0.00520833 | 0.10416667 | — |
| targeted | B-G | 40 | 0.00781250 | 0.10625000 | — |
| targeted | B-G | 80 | 0.00390625 | 0.10312500 | — |
| targeted | B-G | 160 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 320 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 480 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 0 | 0.04427083 | 0.13541667 | — |
| sentinel | B-S | 1 | 0.03515625 | 0.12812500 | — |
| sentinel | B-S | 2 | 0.03515625 | 0.12812500 | — |
| sentinel | B-S | 5 | 0.02864583 | 0.12291667 | — |
| sentinel | B-S | 10 | 0.00390625 | 0.10312500 | — |
| sentinel | B-S | 20 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 40 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 80 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 160 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 320 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 480 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 0 | 0.29557292 | 0.33645833 | — |
| sentinel | B-G | 1 | 0.31250000 | 0.35000000 | — |
| sentinel | B-G | 2 | 0.27604167 | 0.32083333 | — |
| sentinel | B-G | 5 | 0.09375000 | 0.17500000 | — |
| sentinel | B-G | 10 | 0.01302083 | 0.11041667 | — |
| sentinel | B-G | 20 | 0.00651042 | 0.10520833 | — |
| sentinel | B-G | 40 | 0.00260417 | 0.10208333 | — |
| sentinel | B-G | 80 | 0.00260417 | 0.10208333 | — |
| sentinel | B-G | 160 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 320 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 480 | 0.00000000 | 0.10000000 | — |
| maps | B-S | 0 | 0.00000000 | 0.02941176 | 0.00000000 |
| maps | B-S | 1 | 0.00292969 | 0.03216912 | 0.00292969 |
| maps | B-S | 2 | 0.03564453 | 0.06295956 | 0.03564453 |
| maps | B-S | 5 | 0.06262207 | 0.08835018 | 0.06262207 |
| maps | B-S | 10 | 0.07116699 | 0.09639246 | 0.07116699 |
| maps | B-S | 20 | 0.07019043 | 0.09547335 | 0.07019043 |
| maps | B-S | 40 | 0.11657715 | 0.13913143 | 0.11657715 |
| maps | B-S | 80 | 0.16308594 | 0.18290441 | 0.16308594 |
| maps | B-S | 160 | 0.23620605 | 0.25172335 | 0.23620605 |
| maps | B-S | 320 | 0.32568359 | 0.33593750 | 0.32568359 |
| maps | B-S | 480 | 0.37817383 | 0.38534007 | 0.37817383 |
| maps | B-G | 0 | 0.04992676 | 0.07640165 | 0.00000000 |
| maps | B-G | 1 | 0.06518555 | 0.09076287 | 0.01525879 |
| maps | B-G | 2 | 0.05407715 | 0.08030790 | 0.00415039 |
| maps | B-G | 5 | 0.06140137 | 0.08720129 | 0.01147461 |
| maps | B-G | 10 | 0.07128906 | 0.09650735 | 0.02136230 |
| maps | B-G | 20 | 0.06701660 | 0.09248621 | 0.01708984 |
| maps | B-G | 40 | 0.07226562 | 0.09742647 | 0.02233887 |
| maps | B-G | 80 | 0.17175293 | 0.19106158 | 0.12182617 |
| maps | B-G | 160 | 0.17932129 | 0.19818474 | 0.12939453 |
| maps | B-G | 320 | 0.32849121 | 0.33857996 | 0.27856445 |
| maps | B-G | 480 | 0.40368652 | 0.40935202 | 0.35375977 |

### Failure decomposition

Authoritative codes and grouped outcomes are mutually exclusive. Overlapping cap/tag/syntax indicators are printed separately and must not be added as disjoint failures. `output_contract_or_parse` uses the original invalid-tag/lexing/syntax flags; `executable_wrong_target` is the original `wrong_target` code; other well-formed task failures include operand/operation/arithmetic constraints. Conditional output-valid accuracy is shown with its denominator and unconditional score. It does not establish that an invalid output would have been correct.

| Role | Arm | Update | N | Unconditional | Valid successes/N | Conditional | Cap | Bad tag | Bad syntax | Failure union | Mean tokens | Median |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| sentinel | B-S | 0 | 768 | 0.04427083 | 34/400 | 0.08500000 | 23 | 25 | 368 | 368 | 232.82031250 | 87.00000000 |
| targeted | B-S | 0 | 768 | 0.29817708 | 229/519 | 0.44123314 | 14 | 15 | 249 | 249 | 174.16927083 | 79.50000000 |
| sentinel | B-S | 1 | 768 | 0.03515625 | 27/440 | 0.06136364 | 6 | 6 | 328 | 328 | 133.19531250 | 85.00000000 |
| targeted | B-S | 1 | 768 | 0.24609375 | 189/533 | 0.35459662 | 7 | 7 | 235 | 235 | 130.71354167 | 79.00000000 |
| sentinel | B-S | 2 | 768 | 0.03515625 | 27/506 | 0.05335968 | 2 | 2 | 262 | 262 | 100.98437500 | 82.00000000 |
| targeted | B-S | 2 | 768 | 0.16927083 | 130/563 | 0.23090586 | 3 | 3 | 205 | 205 | 100.66015625 | 78.00000000 |
| sentinel | B-S | 5 | 768 | 0.02864583 | 22/559 | 0.03935599 | 6 | 7 | 209 | 209 | 111.62109375 | 72.00000000 |
| targeted | B-S | 5 | 768 | 0.07812500 | 60/570 | 0.10526316 | 7 | 8 | 198 | 198 | 115.12239583 | 71.00000000 |
| sentinel | B-S | 10 | 768 | 0.00390625 | 3/92 | 0.03260870 | 40 | 66 | 676 | 676 | 288.69140625 | 63.00000000 |
| targeted | B-S | 10 | 768 | 0.00520833 | 4/94 | 0.04255319 | 40 | 71 | 674 | 674 | 285.39713542 | 62.00000000 |
| sentinel | B-S | 20 | 768 | 0.00000000 | 0/227 | 0.00000000 | 24 | 30 | 541 | 541 | 179.04427083 | 50.00000000 |
| targeted | B-S | 20 | 768 | 0.00130208 | 1/236 | 0.00423729 | 28 | 37 | 532 | 532 | 203.15625000 | 50.00000000 |
| sentinel | B-S | 40 | 768 | 0.00000000 | 0/378 | 0.00000000 | 12 | 20 | 390 | 390 | 99.92838542 | 19.00000000 |
| targeted | B-S | 40 | 768 | 0.00130208 | 1/406 | 0.00246305 | 8 | 16 | 362 | 362 | 81.16015625 | 17.00000000 |
| stage-b | B-S | 0 | 8192 | 0.00000000 | 0/0 | undefined / unavailable | 820 | 8192 | 8192 | 8192 | 50.20336914 | 44.00000000 |
| stage-b | B-S | 1 | 8192 | 0.00292969 | 24/623 | 0.03852327 | 561 | 930 | 7569 | 7569 | 33.09777832 | 21.00000000 |
| stage-b | B-S | 2 | 8192 | 0.03564453 | 292/7169 | 0.04073092 | 36 | 314 | 1023 | 1023 | 14.47399902 | 13.00000000 |
| stage-b | B-S | 5 | 8192 | 0.06262207 | 513/8024 | 0.06393320 | 0 | 156 | 168 | 168 | 12.86193848 | 13.00000000 |
| stage-b | B-S | 10 | 8192 | 0.07116699 | 583/8192 | 0.07116699 | 0 | 0 | 0 | 0 | 12.80712891 | 13.00000000 |
| stage-b | B-S | 20 | 8192 | 0.07019043 | 575/8192 | 0.07019043 | 0 | 0 | 0 | 0 | 12.68115234 | 13.00000000 |
| stage-b | B-S | 40 | 8192 | 0.11657715 | 955/8192 | 0.11657715 | 0 | 0 | 0 | 0 | 12.66503906 | 13.00000000 |
| sentinel | B-G | 0 | 768 | 0.29557292 | 227/313 | 0.72523962 | 227 | 280 | 455 | 455 | 2011.19010417 | 1559.00000000 |
| targeted | B-G | 0 | 768 | 0.30859375 | 237/328 | 0.72256098 | 229 | 289 | 440 | 440 | 1970.62890625 | 1448.00000000 |
| sentinel | B-G | 1 | 768 | 0.31250000 | 240/317 | 0.75709779 | 221 | 345 | 451 | 451 | 1945.93619792 | 1368.00000000 |
| targeted | B-G | 1 | 768 | 0.28515625 | 219/317 | 0.69085174 | 241 | 350 | 451 | 451 | 1956.21223958 | 1284.00000000 |
| sentinel | B-G | 2 | 768 | 0.27604167 | 212/346 | 0.61271676 | 222 | 309 | 422 | 422 | 1827.67838542 | 1105.50000000 |
| targeted | B-G | 2 | 768 | 0.27213542 | 209/334 | 0.62574850 | 229 | 322 | 434 | 434 | 1829.90494792 | 1105.50000000 |
| sentinel | B-G | 5 | 768 | 0.09375000 | 72/499 | 0.14428858 | 80 | 134 | 269 | 269 | 632.69270833 | 58.00000000 |
| targeted | B-G | 5 | 768 | 0.10416667 | 80/477 | 0.16771488 | 78 | 137 | 291 | 291 | 633.80859375 | 77.00000000 |
| sentinel | B-G | 10 | 768 | 0.01302083 | 10/767 | 0.01303781 | 0 | 0 | 1 | 1 | 21.31770833 | 22.00000000 |
| targeted | B-G | 10 | 768 | 0.00781250 | 6/753 | 0.00796813 | 1 | 1 | 15 | 15 | 26.78385417 | 22.00000000 |
| sentinel | B-G | 20 | 768 | 0.00651042 | 5/719 | 0.00695410 | 0 | 0 | 49 | 49 | 21.35937500 | 22.00000000 |
| targeted | B-G | 20 | 768 | 0.00520833 | 4/711 | 0.00562588 | 0 | 0 | 57 | 57 | 21.59505208 | 22.00000000 |
| sentinel | B-G | 40 | 768 | 0.00260417 | 2/741 | 0.00269906 | 0 | 1 | 27 | 27 | 21.34375000 | 22.00000000 |
| targeted | B-G | 40 | 768 | 0.00781250 | 6/725 | 0.00827586 | 0 | 0 | 43 | 43 | 21.46354167 | 22.00000000 |
| stage-b | B-G | 0 | 8192 | 0.04992676 | 409/6598 | 0.06198848 | 627 | 1164 | 1594 | 1594 | 24.99951172 | 13.00000000 |
| stage-b | B-G | 1 | 8192 | 0.06518555 | 534/8192 | 0.06518555 | 0 | 0 | 0 | 0 | 12.93798828 | 13.00000000 |
| stage-b | B-G | 2 | 8192 | 0.05407715 | 443/8174 | 0.05419623 | 0 | 0 | 18 | 18 | 12.01879883 | 13.00000000 |
| stage-b | B-G | 5 | 8192 | 0.06140137 | 503/8192 | 0.06140137 | 0 | 0 | 0 | 0 | 12.86108398 | 13.00000000 |
| stage-b | B-G | 10 | 8192 | 0.07128906 | 584/8192 | 0.07128906 | 0 | 0 | 0 | 0 | 12.82690430 | 13.00000000 |
| stage-b | B-G | 20 | 8192 | 0.06701660 | 549/8192 | 0.06701660 | 0 | 0 | 0 | 0 | 12.70605469 | 13.00000000 |
| stage-b | B-G | 40 | 8192 | 0.07226562 | 592/8192 | 0.07226562 | 0 | 0 | 0 | 0 | 12.55957031 | 13.00000000 |

| Role | Arm | Update | Disjoint authoritative codes (including correct) |
| --- | --- | --- | --- |
| sentinel | B-S | 0 | {"ast_limit_exceeded": 278, "correct": 34, "invalid_character": 32, "invalid_syntax": 33, "missing_answer_tag": 25, "operand_multiset_mismatch": 235, "wrong_target": 131} |
| targeted | B-S | 0 | {"ast_limit_exceeded": 202, "correct": 229, "invalid_character": 11, "invalid_syntax": 21, "missing_answer_tag": 15, "operand_multiset_mismatch": 213, "wrong_target": 77} |
| sentinel | B-S | 1 | {"ast_limit_exceeded": 266, "correct": 27, "invalid_character": 21, "invalid_syntax": 35, "missing_answer_tag": 6, "operand_multiset_mismatch": 250, "wrong_target": 163} |
| targeted | B-S | 1 | {"ast_limit_exceeded": 185, "correct": 189, "invalid_character": 18, "invalid_syntax": 25, "missing_answer_tag": 7, "operand_multiset_mismatch": 234, "wrong_target": 110} |
| sentinel | B-S | 2 | {"ast_limit_exceeded": 223, "correct": 27, "invalid_character": 19, "invalid_syntax": 18, "missing_answer_tag": 2, "operand_multiset_mismatch": 246, "wrong_target": 233} |
| targeted | B-S | 2 | {"ast_limit_exceeded": 175, "correct": 130, "invalid_character": 10, "invalid_syntax": 17, "missing_answer_tag": 3, "operand_multiset_mismatch": 220, "wrong_target": 213} |
| sentinel | B-S | 5 | {"ast_limit_exceeded": 131, "correct": 22, "invalid_character": 44, "invalid_syntax": 27, "missing_answer_tag": 7, "operand_multiset_mismatch": 259, "wrong_target": 278} |
| targeted | B-S | 5 | {"ast_limit_exceeded": 112, "correct": 60, "invalid_character": 39, "invalid_syntax": 39, "missing_answer_tag": 7, "multiple_answer_tags": 1, "operand_multiset_mismatch": 250, "wrong_target": 260} |
| sentinel | B-S | 10 | {"ast_limit_exceeded": 2, "correct": 3, "invalid_character": 608, "missing_answer_tag": 38, "multiple_answer_tags": 28, "operand_multiset_mismatch": 81, "wrong_target": 8} |
| targeted | B-S | 10 | {"answer_too_long": 3, "ast_limit_exceeded": 4, "correct": 4, "invalid_character": 595, "invalid_syntax": 1, "missing_answer_tag": 35, "multiple_answer_tags": 36, "operand_multiset_mismatch": 85, "wrong_target": 5} |
| sentinel | B-S | 20 | {"answer_too_long": 1, "ast_limit_exceeded": 2, "invalid_character": 508, "missing_answer_tag": 18, "multiple_answer_tags": 12, "operand_multiset_mismatch": 217, "wrong_target": 10} |
| targeted | B-S | 20 | {"ast_limit_exceeded": 1, "correct": 1, "invalid_character": 494, "missing_answer_tag": 26, "multiple_answer_tags": 11, "operand_multiset_mismatch": 221, "wrong_target": 14} |
| sentinel | B-S | 40 | {"ast_limit_exceeded": 1, "invalid_character": 369, "missing_answer_tag": 11, "multiple_answer_tags": 9, "operand_multiset_mismatch": 364, "wrong_target": 14} |
| targeted | B-S | 40 | {"answer_too_long": 1, "ast_limit_exceeded": 4, "correct": 1, "invalid_character": 340, "invalid_syntax": 1, "missing_answer_tag": 7, "multiple_answer_tags": 9, "operand_multiset_mismatch": 396, "wrong_target": 9} |
| stage-b | B-S | 0 | {"invalid_program": 4235, "missing_answer_tag": 3910, "multiple_answer_tags": 47} |
| stage-b | B-S | 1 | {"correct": 24, "invalid_program": 4632, "missing_answer_tag": 536, "multiple_answer_tags": 195, "program_too_long": 2570, "wrong_target": 235} |
| stage-b | B-S | 2 | {"correct": 292, "illegal_instruction": 15, "invalid_program": 466, "missing_answer_tag": 115, "multiple_answer_tags": 2, "program_too_long": 1491, "wrong_target": 5811} |
| stage-b | B-S | 5 | {"correct": 513, "illegal_instruction": 45, "invalid_program": 10, "missing_answer_tag": 153, "program_too_long": 15, "wrong_target": 7456} |
| stage-b | B-S | 10 | {"correct": 583, "illegal_instruction": 1, "wrong_target": 7608} |
| stage-b | B-S | 20 | {"correct": 575, "illegal_instruction": 2, "wrong_target": 7615} |
| stage-b | B-S | 40 | {"correct": 955, "illegal_instruction": 925, "wrong_target": 6312} |
| sentinel | B-G | 0 | {"ast_limit_exceeded": 20, "correct": 227, "empty_answer": 3, "invalid_character": 151, "invalid_syntax": 1, "missing_answer_tag": 268, "multiple_answer_tags": 12, "operand_multiset_mismatch": 43, "wrong_target": 43} |
| targeted | B-G | 0 | {"ast_limit_exceeded": 23, "correct": 237, "empty_answer": 2, "invalid_character": 120, "invalid_syntax": 6, "missing_answer_tag": 280, "multiple_answer_tags": 9, "operand_multiset_mismatch": 47, "wrong_target": 44} |
| sentinel | B-G | 1 | {"ast_limit_exceeded": 18, "correct": 240, "empty_answer": 1, "invalid_character": 85, "invalid_syntax": 2, "missing_answer_tag": 334, "multiple_answer_tags": 11, "operand_multiset_mismatch": 39, "wrong_target": 38} |
| targeted | B-G | 1 | {"ast_limit_exceeded": 9, "correct": 219, "empty_answer": 2, "invalid_character": 86, "invalid_syntax": 4, "missing_answer_tag": 344, "multiple_answer_tags": 6, "operand_multiset_mismatch": 59, "wrong_target": 39} |
| sentinel | B-G | 2 | {"ast_limit_exceeded": 28, "correct": 212, "empty_answer": 1, "invalid_character": 80, "invalid_syntax": 4, "missing_answer_tag": 303, "multiple_answer_tags": 6, "operand_multiset_mismatch": 78, "wrong_target": 56} |
| targeted | B-G | 2 | {"ast_limit_exceeded": 22, "correct": 209, "empty_answer": 1, "invalid_character": 87, "invalid_syntax": 2, "missing_answer_tag": 317, "multiple_answer_tags": 5, "operand_multiset_mismatch": 74, "wrong_target": 51} |
| sentinel | B-G | 5 | {"ast_limit_exceeded": 34, "correct": 72, "division_by_zero": 2, "invalid_character": 94, "invalid_syntax": 7, "missing_answer_tag": 133, "multiple_answer_tags": 1, "operand_multiset_mismatch": 141, "wrong_target": 284} |
| targeted | B-G | 5 | {"ast_limit_exceeded": 33, "correct": 80, "empty_answer": 1, "invalid_character": 107, "invalid_syntax": 13, "missing_answer_tag": 136, "multiple_answer_tags": 1, "operand_multiset_mismatch": 137, "wrong_target": 260} |
| sentinel | B-G | 10 | {"correct": 10, "division_by_zero": 1, "invalid_syntax": 1, "operand_multiset_mismatch": 195, "wrong_target": 561} |
| targeted | B-G | 10 | {"ast_limit_exceeded": 2, "correct": 6, "invalid_character": 1, "invalid_syntax": 11, "missing_answer_tag": 1, "operand_multiset_mismatch": 191, "wrong_target": 556} |
| sentinel | B-G | 20 | {"ast_limit_exceeded": 9, "correct": 5, "invalid_character": 1, "invalid_syntax": 39, "operand_multiset_mismatch": 271, "wrong_target": 443} |
| targeted | B-G | 20 | {"ast_limit_exceeded": 9, "correct": 4, "invalid_syntax": 48, "operand_multiset_mismatch": 282, "wrong_target": 425} |
| sentinel | B-G | 40 | {"ast_limit_exceeded": 15, "correct": 2, "invalid_character": 4, "invalid_syntax": 7, "missing_answer_tag": 1, "operand_multiset_mismatch": 334, "wrong_target": 405} |
| targeted | B-G | 40 | {"ast_limit_exceeded": 24, "correct": 6, "invalid_character": 1, "invalid_syntax": 18, "operand_multiset_mismatch": 346, "wrong_target": 373} |
| stage-b | B-G | 0 | {"correct": 409, "illegal_instruction": 118, "invalid_program": 535, "missing_answer_tag": 758, "multiple_answer_tags": 2, "program_too_long": 775, "wrong_target": 5595} |
| stage-b | B-G | 1 | {"correct": 534, "illegal_instruction": 1, "wrong_target": 7657} |
| stage-b | B-G | 2 | {"correct": 443, "invalid_program": 18, "wrong_target": 7731} |
| stage-b | B-G | 5 | {"correct": 503, "wrong_target": 7689} |
| stage-b | B-G | 10 | {"correct": 584, "wrong_target": 7608} |
| stage-b | B-G | 20 | {"correct": 549, "wrong_target": 7643} |
| stage-b | B-G | 40 | {"correct": 592, "wrong_target": 7600} |

## Pair 2

### Absolute performance, gain, and endpoint

| Arm | Baseline | Absolute AUC 0–40 | Gain AUC 0–40 | Absolute AUC 0–480 | Gain AUC 0–480 | Endpoint 480 |
| --- | --- | --- | --- | --- | --- | --- |
| B-S | 0.00000000 | 0.06969604 | 0.06969604 | 0.12707977 | 0.12707977 | 0.26855469 |
| B-G | 0.05346680 | 0.07224579 | 0.01877899 | 0.24099058 | 0.18752378 | 0.36865234 |

Gain contrast = absolute-AUC contrast − baseline contrast:

| Window | Absolute difference | Baseline difference | Gain difference | Numerical residual |
| --- | --- | --- | --- | --- |
| [0, 40] | 0.00254974 | 0.05346680 | -0.05091705 | 0 |
| [0, 480] | 0.11391080 | 0.05346680 | 0.06044401 | 0 |

### Unconditional retention

| Arm | Role | Raw baseline | Absolute AUC 0–20 | Own-baseline-relative AUC | Half-life | Bracket/status |
| --- | --- | --- | --- | --- | --- | --- |
| B-S | targeted | 0.17578125 | 0.01438802 | 0.08185185 | 1.13636364 | [1, 2] |
| B-S | sentinel | 0.05989583 | 0.01064453 | 0.17771739 | 1.68421053 | [1, 2] |
| B-G | targeted | 0.19791667 | 0.04820964 | 0.24358553 | 3.34328358 | [2, 5] |
| B-G | sentinel | 0.19791667 | 0.04833984 | 0.24424342 | 3.50000000 | [2, 5] |

These scores are total observed pre-B performance, not survival restricted to newly acquired items.

### Paired item uncertainty

| Contrast | Estimate | 95% lower | 95% upper | Items |
| --- | --- | --- | --- | --- |
| pair2-seed29-targeted-absolute-AUC-0-20-B-G-minus-B-S | 0.03382161 | 0.02727865 | 0.04059245 | 192 |
| pair2-seed29-sentinel-absolute-AUC-0-20-B-G-minus-B-S | 0.03769531 | 0.03033854 | 0.04547526 | 192 |
| pair2-seed29-maps-absolute-AUC-0-40-B-G-minus-B-S | 0.00254974 | 0.00033875 | 0.00474396 | 512 |
| pair2-seed29-maps-gain-AUC-0-40-B-G-minus-B-S | -0.05091705 | -0.05740822 | -0.04452816 | 512 |
| pair2-seed29-maps-absolute-AUC-0-480-B-G-minus-B-S | 0.11391080 | 0.10146001 | 0.12645001 | 512 |
| pair2-seed29-maps-gain-AUC-0-480-B-G-minus-B-S | 0.06044401 | 0.04813015 | 0.07315776 | 512 |
| pair2-maps-baseline-B-G-minus-B-S | 0.05346680 | 0.04736328 | 0.05981445 | 512 |
| pair2-maps-endpoint480-B-G-minus-B-S | 0.10009766 | 0.08264160 | 0.11804199 | 512 |

### First attainment at comparable absolute MAPS performance

| Threshold | Arm | Status | Interpolated update | Observed bracket | Targeted TCES | Eligible B-G−B-S |
| --- | --- | --- | --- | --- | --- | --- |
| 0.06 | B-S | crossed | 2.99620690 | [2, 5] | 0.01912356 | — |
| 0.06 | B-G | crossed | 0.27875000 | [0, 1] | 0.19864258 | 0.17951901 |
| 0.07 | B-S | crossed | 5.76551724 | [5, 10] | 0.00500898 | — |
| 0.07 | B-G | crossed | 0.70541667 | [0, 1] | 0.19975369 | 0.19474471 |
| 0.1 | B-S | crossed | 226.08829175 | [160, 320] | 0.00000000 | — |
| 0.1 | B-G | crossed | 55.27984344 | [40, 80] | 0.00149217 | 0.00149217 |
| 0.2 | B-S | crossed | 396.41302326 | [320, 480] | 0.00000000 | — |
| 0.2 | B-G | crossed | 188.76073620 | [160, 320] | 0.00000000 | 0.00000000 |

`baseline_exceeded` is not equivalent learning progress. A two-arm post-training difference is printed only when both arms have a first crossing from below. Interpolation uses the same adjacent interval for update and retention and never extrapolates or erases a reversal.

### Full raw and posterior trajectories

| Panel/role | Arm | Update | Raw Pass@1 | Jeffreys posterior | MAPS raw gain |
| --- | --- | --- | --- | --- | --- |
| targeted | B-S | 0 | 0.17578125 | 0.24062500 | — |
| targeted | B-S | 1 | 0.09765625 | 0.17812500 | — |
| targeted | B-S | 2 | 0.02604167 | 0.12083333 | — |
| targeted | B-S | 5 | 0.00520833 | 0.10416667 | — |
| targeted | B-S | 10 | 0.00390625 | 0.10312500 | — |
| targeted | B-S | 20 | 0.00000000 | 0.10000000 | — |
| targeted | B-S | 40 | 0.00000000 | 0.10000000 | — |
| targeted | B-S | 80 | 0.00000000 | 0.10000000 | — |
| targeted | B-S | 160 | 0.00000000 | 0.10000000 | — |
| targeted | B-S | 320 | 0.00000000 | 0.10000000 | — |
| targeted | B-S | 480 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 0 | 0.19791667 | 0.25833333 | — |
| targeted | B-G | 1 | 0.20052083 | 0.26041667 | — |
| targeted | B-G | 2 | 0.13802083 | 0.21041667 | — |
| targeted | B-G | 5 | 0.05078125 | 0.14062500 | — |
| targeted | B-G | 10 | 0.02473958 | 0.11979167 | — |
| targeted | B-G | 20 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 40 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 80 | 0.00390625 | 0.10312500 | — |
| targeted | B-G | 160 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 320 | 0.00000000 | 0.10000000 | — |
| targeted | B-G | 480 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 0 | 0.05989583 | 0.14791667 | — |
| sentinel | B-S | 1 | 0.04687500 | 0.13750000 | — |
| sentinel | B-S | 2 | 0.02213542 | 0.11770833 | — |
| sentinel | B-S | 5 | 0.01562500 | 0.11250000 | — |
| sentinel | B-S | 10 | 0.00390625 | 0.10312500 | — |
| sentinel | B-S | 20 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 40 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 80 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 160 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 320 | 0.00000000 | 0.10000000 | — |
| sentinel | B-S | 480 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 0 | 0.19791667 | 0.25833333 | — |
| sentinel | B-G | 1 | 0.20182292 | 0.26145833 | — |
| sentinel | B-G | 2 | 0.15885417 | 0.22708333 | — |
| sentinel | B-G | 5 | 0.03906250 | 0.13125000 | — |
| sentinel | B-G | 10 | 0.02473958 | 0.11979167 | — |
| sentinel | B-G | 20 | 0.00130208 | 0.10104167 | — |
| sentinel | B-G | 40 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 80 | 0.00651042 | 0.10520833 | — |
| sentinel | B-G | 160 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 320 | 0.00000000 | 0.10000000 | — |
| sentinel | B-G | 480 | 0.00000000 | 0.10000000 | — |
| maps | B-S | 0 | 0.00000000 | 0.02941176 | 0.00000000 |
| maps | B-S | 1 | 0.00317383 | 0.03239890 | 0.00317383 |
| maps | B-S | 2 | 0.05529785 | 0.08145680 | 0.05529785 |
| maps | B-S | 5 | 0.06945801 | 0.09478401 | 0.06945801 |
| maps | B-S | 10 | 0.07299805 | 0.09811581 | 0.07299805 |
| maps | B-S | 20 | 0.07165527 | 0.09685202 | 0.07165527 |
| maps | B-S | 40 | 0.07739258 | 0.10225184 | 0.07739258 |
| maps | B-S | 80 | 0.07263184 | 0.09777114 | 0.07263184 |
| maps | B-S | 160 | 0.07373047 | 0.09880515 | 0.07373047 |
| maps | B-S | 320 | 0.13732910 | 0.15866268 | 0.13732910 |
| maps | B-S | 480 | 0.26855469 | 0.28216912 | 0.26855469 |
| maps | B-G | 0 | 0.05346680 | 0.07973346 | 0.00000000 |
| maps | B-G | 1 | 0.07690430 | 0.10179228 | 0.02343750 |
| maps | B-G | 2 | 0.04992676 | 0.07640165 | -0.00354004 |
| maps | B-G | 5 | 0.06848145 | 0.09386489 | 0.01501465 |
| maps | B-G | 10 | 0.07019043 | 0.09547335 | 0.01672363 |
| maps | B-G | 20 | 0.07495117 | 0.09995404 | 0.02148438 |
| maps | B-G | 40 | 0.07617188 | 0.10110294 | 0.02270508 |
| maps | B-G | 80 | 0.13854980 | 0.15981158 | 0.08508301 |
| maps | B-G | 160 | 0.17138672 | 0.19071691 | 0.11791992 |
| maps | B-G | 320 | 0.33056641 | 0.34053309 | 0.27709961 |
| maps | B-G | 480 | 0.36865234 | 0.37637868 | 0.31518555 |

### Failure decomposition

Authoritative codes and grouped outcomes are mutually exclusive. Overlapping cap/tag/syntax indicators are printed separately and must not be added as disjoint failures. `output_contract_or_parse` uses the original invalid-tag/lexing/syntax flags; `executable_wrong_target` is the original `wrong_target` code; other well-formed task failures include operand/operation/arithmetic constraints. Conditional output-valid accuracy is shown with its denominator and unconditional score. It does not establish that an invalid output would have been correct.

| Role | Arm | Update | N | Unconditional | Valid successes/N | Conditional | Cap | Bad tag | Bad syntax | Failure union | Mean tokens | Median |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| targeted | B-S | 0 | 768 | 0.17578125 | 135/359 | 0.37604457 | 133 | 135 | 409 | 409 | 814.52213542 | 96.00000000 |
| sentinel | B-S | 0 | 768 | 0.05989583 | 46/308 | 0.14935065 | 154 | 161 | 460 | 460 | 936.05078125 | 101.50000000 |
| targeted | B-S | 1 | 768 | 0.09765625 | 75/591 | 0.12690355 | 15 | 33 | 177 | 177 | 160.03125000 | 68.00000000 |
| sentinel | B-S | 1 | 768 | 0.04687500 | 36/614 | 0.05863192 | 11 | 25 | 154 | 154 | 127.47265625 | 68.00000000 |
| targeted | B-S | 2 | 768 | 0.02604167 | 20/539 | 0.03710575 | 7 | 47 | 229 | 229 | 81.28906250 | 47.00000000 |
| sentinel | B-S | 2 | 768 | 0.02213542 | 17/539 | 0.03153989 | 9 | 50 | 229 | 229 | 91.77994792 | 38.50000000 |
| targeted | B-S | 5 | 768 | 0.00520833 | 4/451 | 0.00886918 | 29 | 64 | 317 | 317 | 202.11848958 | 51.00000000 |
| sentinel | B-S | 5 | 768 | 0.01562500 | 12/468 | 0.02564103 | 21 | 52 | 300 | 300 | 159.43619792 | 50.00000000 |
| targeted | B-S | 10 | 768 | 0.00390625 | 3/402 | 0.00746269 | 33 | 70 | 366 | 366 | 224.87500000 | 22.00000000 |
| sentinel | B-S | 10 | 768 | 0.00390625 | 3/432 | 0.00694444 | 23 | 54 | 336 | 336 | 169.10026042 | 16.00000000 |
| targeted | B-S | 20 | 768 | 0.00000000 | 0/359 | 0.00000000 | 18 | 66 | 409 | 409 | 138.32812500 | 16.00000000 |
| sentinel | B-S | 20 | 768 | 0.00000000 | 0/416 | 0.00000000 | 14 | 40 | 352 | 352 | 104.30729167 | 15.00000000 |
| targeted | B-S | 40 | 768 | 0.00000000 | 0/465 | 0.00000000 | 0 | 28 | 303 | 303 | 25.89583333 | 13.00000000 |
| sentinel | B-S | 40 | 768 | 0.00000000 | 0/504 | 0.00000000 | 4 | 25 | 264 | 264 | 41.11328125 | 12.00000000 |
| stage-b | B-S | 0 | 8192 | 0.00000000 | 0/0 | undefined / unavailable | 1640 | 8192 | 8192 | 8192 | 61.36071777 | 47.00000000 |
| stage-b | B-S | 1 | 8192 | 0.00317383 | 26/698 | 0.03724928 | 1301 | 1680 | 7494 | 7494 | 42.51708984 | 22.00000000 |
| stage-b | B-S | 2 | 8192 | 0.05529785 | 453/7583 | 0.05973889 | 4 | 154 | 609 | 609 | 12.39343262 | 13.00000000 |
| stage-b | B-S | 5 | 8192 | 0.06945801 | 569/8181 | 0.06955140 | 0 | 0 | 11 | 11 | 12.86474609 | 13.00000000 |
| stage-b | B-S | 10 | 8192 | 0.07299805 | 598/8192 | 0.07299805 | 0 | 0 | 0 | 0 | 12.88500977 | 13.00000000 |
| stage-b | B-S | 20 | 8192 | 0.07165527 | 587/8192 | 0.07165527 | 0 | 0 | 0 | 0 | 12.47119141 | 13.00000000 |
| stage-b | B-S | 40 | 8192 | 0.07739258 | 634/8192 | 0.07739258 | 0 | 0 | 0 | 0 | 12.65405273 | 13.00000000 |
| targeted | B-G | 0 | 768 | 0.19791667 | 152/303 | 0.50165017 | 214 | 271 | 465 | 465 | 1814.83203125 | 1122.00000000 |
| sentinel | B-G | 0 | 768 | 0.19791667 | 152/269 | 0.56505576 | 230 | 305 | 499 | 499 | 1805.39453125 | 1031.00000000 |
| targeted | B-G | 1 | 768 | 0.20052083 | 154/251 | 0.61354582 | 240 | 381 | 517 | 517 | 1935.78385417 | 1319.00000000 |
| sentinel | B-G | 1 | 768 | 0.20182292 | 155/249 | 0.62248996 | 250 | 385 | 519 | 519 | 1895.97526042 | 1156.00000000 |
| targeted | B-G | 2 | 768 | 0.13802083 | 106/255 | 0.41568627 | 183 | 345 | 513 | 513 | 1442.52604167 | 453.50000000 |
| sentinel | B-G | 2 | 768 | 0.15885417 | 122/261 | 0.46743295 | 204 | 357 | 507 | 507 | 1584.82552083 | 642.00000000 |
| targeted | B-G | 5 | 768 | 0.05078125 | 39/360 | 0.10833333 | 76 | 186 | 408 | 408 | 612.29557292 | 94.00000000 |
| sentinel | B-G | 5 | 768 | 0.03906250 | 30/341 | 0.08797654 | 80 | 194 | 427 | 427 | 590.32682292 | 98.50000000 |
| targeted | B-G | 10 | 768 | 0.02473958 | 19/588 | 0.03231293 | 24 | 47 | 180 | 180 | 210.42838542 | 24.00000000 |
| sentinel | B-G | 10 | 768 | 0.02473958 | 19/563 | 0.03374778 | 28 | 61 | 205 | 205 | 237.78385417 | 24.00000000 |
| targeted | B-G | 20 | 768 | 0.00000000 | 0/677 | 0.00000000 | 1 | 2 | 91 | 91 | 23.31640625 | 12.00000000 |
| sentinel | B-G | 20 | 768 | 0.00130208 | 1/697 | 0.00143472 | 0 | 0 | 71 | 71 | 13.96223958 | 12.00000000 |
| targeted | B-G | 40 | 768 | 0.00000000 | 0/736 | 0.00000000 | 0 | 0 | 32 | 32 | 13.49739583 | 12.00000000 |
| sentinel | B-G | 40 | 768 | 0.00000000 | 0/731 | 0.00000000 | 0 | 0 | 37 | 37 | 13.18489583 | 12.00000000 |
| stage-b | B-G | 0 | 8192 | 0.05346680 | 438/6603 | 0.06633348 | 573 | 1119 | 1589 | 1589 | 24.33386230 | 13.00000000 |
| stage-b | B-G | 1 | 8192 | 0.07690430 | 630/8192 | 0.07690430 | 0 | 0 | 0 | 0 | 12.93664551 | 13.00000000 |
| stage-b | B-G | 2 | 8192 | 0.04992676 | 409/8119 | 0.05037566 | 0 | 0 | 73 | 73 | 11.72827148 | 13.00000000 |
| stage-b | B-G | 5 | 8192 | 0.06848145 | 561/7925 | 0.07078864 | 0 | 0 | 267 | 267 | 12.82702637 | 13.00000000 |
| stage-b | B-G | 10 | 8192 | 0.07019043 | 575/8192 | 0.07019043 | 0 | 0 | 0 | 0 | 12.57495117 | 13.00000000 |
| stage-b | B-G | 20 | 8192 | 0.07495117 | 614/8192 | 0.07495117 | 0 | 0 | 0 | 0 | 12.67285156 | 13.00000000 |
| stage-b | B-G | 40 | 8192 | 0.07617188 | 624/8192 | 0.07617188 | 0 | 0 | 0 | 0 | 12.52050781 | 13.00000000 |

| Role | Arm | Update | Disjoint authoritative codes (including correct) |
| --- | --- | --- | --- |
| targeted | B-S | 0 | {"ast_limit_exceeded": 175, "correct": 135, "invalid_character": 35, "invalid_syntax": 64, "missing_answer_tag": 135, "operand_multiset_mismatch": 137, "wrong_target": 87} |
| sentinel | B-S | 0 | {"ast_limit_exceeded": 195, "correct": 46, "invalid_character": 30, "invalid_syntax": 74, "missing_answer_tag": 161, "operand_multiset_mismatch": 156, "wrong_target": 106} |
| targeted | B-S | 1 | {"ast_limit_exceeded": 57, "correct": 75, "invalid_character": 48, "invalid_syntax": 39, "missing_answer_tag": 14, "multiple_answer_tags": 19, "operand_multiset_mismatch": 116, "wrong_target": 400} |
| sentinel | B-S | 1 | {"ast_limit_exceeded": 45, "correct": 36, "invalid_character": 51, "invalid_syntax": 33, "missing_answer_tag": 7, "multiple_answer_tags": 18, "operand_multiset_mismatch": 117, "wrong_target": 461} |
| targeted | B-S | 2 | {"ast_limit_exceeded": 6, "correct": 20, "invalid_character": 167, "invalid_syntax": 9, "missing_answer_tag": 3, "multiple_answer_tags": 44, "operand_multiset_mismatch": 244, "wrong_target": 275} |
| sentinel | B-S | 2 | {"ast_limit_exceeded": 6, "correct": 17, "invalid_character": 163, "invalid_syntax": 10, "missing_answer_tag": 6, "multiple_answer_tags": 44, "operand_multiset_mismatch": 249, "wrong_target": 273} |
| targeted | B-S | 5 | {"ast_limit_exceeded": 12, "correct": 4, "invalid_character": 228, "invalid_syntax": 13, "missing_answer_tag": 28, "multiple_answer_tags": 36, "operand_multiset_mismatch": 313, "wrong_target": 134} |
| sentinel | B-S | 5 | {"ast_limit_exceeded": 7, "correct": 12, "invalid_character": 227, "invalid_syntax": 14, "missing_answer_tag": 21, "multiple_answer_tags": 31, "operand_multiset_mismatch": 310, "wrong_target": 146} |
| targeted | B-S | 10 | {"ast_limit_exceeded": 8, "correct": 3, "invalid_character": 281, "invalid_syntax": 7, "missing_answer_tag": 29, "multiple_answer_tags": 41, "operand_multiset_mismatch": 350, "wrong_target": 49} |
| sentinel | B-S | 10 | {"ast_limit_exceeded": 8, "correct": 3, "invalid_character": 267, "invalid_syntax": 7, "missing_answer_tag": 22, "multiple_answer_tags": 32, "operand_multiset_mismatch": 382, "wrong_target": 47} |
| targeted | B-S | 20 | {"answer_too_long": 1, "ast_limit_exceeded": 2, "invalid_character": 335, "invalid_syntax": 5, "missing_answer_tag": 7, "multiple_answer_tags": 59, "operand_multiset_mismatch": 348, "wrong_target": 11} |
| sentinel | B-S | 20 | {"ast_limit_exceeded": 1, "invalid_character": 310, "invalid_syntax": 1, "missing_answer_tag": 8, "multiple_answer_tags": 32, "operand_multiset_mismatch": 406, "wrong_target": 10} |
| targeted | B-S | 40 | {"invalid_character": 273, "invalid_syntax": 2, "multiple_answer_tags": 28, "operand_multiset_mismatch": 463, "wrong_target": 2} |
| sentinel | B-S | 40 | {"invalid_character": 235, "invalid_syntax": 4, "missing_answer_tag": 1, "multiple_answer_tags": 24, "operand_multiset_mismatch": 502, "wrong_target": 2} |
| stage-b | B-S | 0 | {"invalid_program": 4832, "missing_answer_tag": 3356, "multiple_answer_tags": 4} |
| stage-b | B-S | 1 | {"correct": 26, "illegal_instruction": 4, "invalid_program": 3899, "missing_answer_tag": 1291, "multiple_answer_tags": 389, "program_too_long": 2227, "wrong_target": 356} |
| stage-b | B-S | 2 | {"correct": 453, "empty_answer": 2, "illegal_instruction": 147, "invalid_program": 478, "missing_answer_tag": 109, "multiple_answer_tags": 15, "program_too_long": 6, "wrong_target": 6982} |
| stage-b | B-S | 5 | {"correct": 569, "illegal_instruction": 44, "invalid_program": 11, "wrong_target": 7568} |
| stage-b | B-S | 10 | {"correct": 598, "illegal_instruction": 1, "wrong_target": 7593} |
| stage-b | B-S | 20 | {"correct": 587, "wrong_target": 7605} |
| stage-b | B-S | 40 | {"correct": 634, "wrong_target": 7558} |
| targeted | B-G | 0 | {"answer_too_long": 2, "ast_limit_exceeded": 19, "correct": 152, "empty_answer": 3, "invalid_character": 166, "invalid_syntax": 4, "missing_answer_tag": 259, "multiple_answer_tags": 12, "operand_multiset_mismatch": 78, "wrong_target": 73} |
| sentinel | B-G | 0 | {"answer_too_long": 1, "ast_limit_exceeded": 30, "correct": 152, "empty_answer": 4, "invalid_character": 152, "invalid_syntax": 7, "missing_answer_tag": 289, "multiple_answer_tags": 16, "operand_multiset_mismatch": 66, "wrong_target": 51} |
| targeted | B-G | 1 | {"ast_limit_exceeded": 25, "correct": 154, "invalid_character": 108, "invalid_syntax": 3, "missing_answer_tag": 378, "multiple_answer_tags": 3, "operand_multiset_mismatch": 55, "wrong_target": 42} |
| sentinel | B-G | 1 | {"ast_limit_exceeded": 22, "correct": 155, "empty_answer": 2, "invalid_character": 102, "invalid_syntax": 8, "missing_answer_tag": 377, "multiple_answer_tags": 8, "operand_multiset_mismatch": 60, "wrong_target": 34} |
| targeted | B-G | 2 | {"ast_limit_exceeded": 32, "correct": 106, "empty_answer": 3, "invalid_character": 128, "invalid_syntax": 5, "missing_answer_tag": 344, "multiple_answer_tags": 1, "operand_multiset_mismatch": 94, "wrong_target": 55} |
| sentinel | B-G | 2 | {"ast_limit_exceeded": 33, "correct": 122, "empty_answer": 3, "invalid_character": 103, "invalid_syntax": 11, "missing_answer_tag": 356, "multiple_answer_tags": 1, "operand_multiset_mismatch": 85, "wrong_target": 54} |
| targeted | B-G | 5 | {"ast_limit_exceeded": 51, "correct": 39, "invalid_character": 161, "invalid_syntax": 10, "missing_answer_tag": 185, "multiple_answer_tags": 1, "operand_multiset_mismatch": 148, "wrong_target": 173} |
| sentinel | B-G | 5 | {"ast_limit_exceeded": 58, "correct": 30, "invalid_character": 159, "invalid_syntax": 16, "missing_answer_tag": 193, "multiple_answer_tags": 1, "operand_multiset_mismatch": 151, "wrong_target": 160} |
| targeted | B-G | 10 | {"ast_limit_exceeded": 14, "correct": 19, "invalid_character": 104, "invalid_syntax": 15, "missing_answer_tag": 47, "operand_multiset_mismatch": 307, "wrong_target": 262} |
| sentinel | B-G | 10 | {"ast_limit_exceeded": 15, "correct": 19, "empty_answer": 2, "invalid_character": 107, "invalid_syntax": 20, "missing_answer_tag": 60, "multiple_answer_tags": 1, "operand_multiset_mismatch": 329, "wrong_target": 215} |
| targeted | B-G | 20 | {"invalid_character": 82, "invalid_syntax": 7, "missing_answer_tag": 1, "multiple_answer_tags": 1, "operand_multiset_mismatch": 655, "wrong_target": 22} |
| sentinel | B-G | 20 | {"correct": 1, "invalid_character": 59, "invalid_syntax": 12, "operand_multiset_mismatch": 678, "wrong_target": 18} |
| targeted | B-G | 40 | {"invalid_character": 26, "invalid_syntax": 6, "operand_multiset_mismatch": 726, "wrong_target": 10} |
| sentinel | B-G | 40 | {"invalid_character": 22, "invalid_syntax": 15, "operand_multiset_mismatch": 724, "wrong_target": 7} |
| stage-b | B-G | 0 | {"correct": 438, "illegal_instruction": 103, "invalid_program": 531, "missing_answer_tag": 723, "multiple_answer_tags": 1, "program_too_long": 846, "wrong_target": 5550} |
| stage-b | B-G | 1 | {"correct": 630, "illegal_instruction": 3, "wrong_target": 7559} |
| stage-b | B-G | 2 | {"correct": 409, "illegal_instruction": 2, "invalid_program": 73, "wrong_target": 7708} |
| stage-b | B-G | 5 | {"correct": 561, "illegal_instruction": 6, "invalid_program": 267, "wrong_target": 7358} |
| stage-b | B-G | 10 | {"correct": 575, "illegal_instruction": 4, "wrong_target": 7613} |
| stage-b | B-G | 20 | {"correct": 614, "wrong_target": 7578} |
| stage-b | B-G | 40 | {"correct": 624, "illegal_instruction": 5, "wrong_target": 7563} |

## Audit conclusion

Pair 1: B-G−B-S targeted raw retention AUC 0–20 is 0.01966146; MAPS absolute AUC 0–480 difference is -0.01056824, whereas its gain-AUC difference is -0.06049500. In the early 0–40 window, absolute and gain differences are -0.00926514 and -0.05919189, with baseline difference 0.04992676. At update 2, output-valid targeted success is B-S 130/563 and B-G 209/334; invalid outputs remain failures in the primary raw scores.

Pair 1: B-S output-valid targeted completions increase from 519/768 at baseline to 563/768 at update 2, while verified successes decrease from 229 to 130. Thus this early unconditional decline is not solely an increase in output-format failures. Later failure composition differs by arm, as retained in the full code table; this does not imply preservation or loss of an unobserved latent capability.

Pair 2: B-G−B-S targeted raw retention AUC 0–20 is 0.03382161; MAPS absolute AUC 0–480 difference is 0.11391080, whereas its gain-AUC difference is 0.06044401. In the early 0–40 window, absolute and gain differences are 0.00254974 and -0.05091705, with baseline difference 0.05346680. At update 2, output-valid targeted success is B-S 20/539 and B-G 106/255; invalid outputs remain failures in the primary raw scores.

Pair 2: B-S output-valid targeted completions increase from 359/768 at baseline to 539/768 at update 2, while verified successes decrease from 135 to 20. Thus this early unconditional decline is not solely an increase in output-format failures. Later failure composition differs by arm, as retained in the full code table; this does not imply preservation or loss of an unobserved latent capability.

What survives: B-G has the larger unconditional targeted retention AUC over 0–20 and the longer targeted half-life in both pairs. What weakens: the early B-S MAPS gain advantage is much smaller on absolute performance in Pair 1, and reverses direction on absolute performance in Pair 2. The 0–480 absolute-AUC contrast has opposite signs across pairs. At first attainment of 0.10 or 0.20 absolute MAPS success, targeted retention is already near zero in both pairs; all these rows remain in the table. Larger gain from a lower starting point is not an absolute-performance advantage, and these observations do not establish a uniformly better retention–learning tradeoff.

Task-validity conditioning changes the denominator and cannot identify hidden arithmetic capability or repair invalid responses. The planned controlled replay can compare the bundled trace-source intervention under a shared supervised acquisition recipe and shared prompts; it cannot isolate an RL objective effect, separate trace content from length/format/strategy, or establish general plasticity from two reused source blocks.

## Reproduction and exclusions

`per-item-counts.jsonl` retains all item trajectories and their realized draw rewards in sample-index order. `figure-data.json` is the exact data used for the figures; `summary.json` retains numerical definitions, old-interval reproduction, new seeds, and all failure denominators. The private source index is saved separately under ignored `runs/replay-v1/`; raw generations and verifier decisions remain in the immutable original run directories. No sealed tests, new completions, or alternative verifier were used.

Rebuild into a new destination from the repository root:

```sh
PYTHONPATH=src uv run python -m duraseed.replay_analysis_pilot --output artifacts/replay-v1/pilot-audit-reproduction
```

Data unavailable: none for the requested Pilot item counts and raw early-task failure decomposition.
